In [1]:
# Run this to disable ligatures in the jupyter notebook
from IPython.core.display import HTML
HTML("""
    <style>    body { font-feature-settings: "liga" 0; }    </style>
""")

# Benchmarking the FHWA Asphalt Framework

Antelope is a complete (though not yet fully comprehensive) LCA computing environment, but the first task it was created for is benchmarking of existing datasets. Today we are going to look at a Federal LCA Commons dataset that is the product of years of multi-stakeholder work across industry, academia, and state and federal government agencies: the Hot Mix Asphalt LCA Framework model prepared under the Federal Highway Administration's (FHWA) Sustainable Pavement Project.  Read all about the Asphalt Framework [here](https://rosap.ntl.bts.gov/view/dot/38470) (if you prefer 15-MB government project reports) or [here](https://link.springer.com/article/10.1007/s11367-020-01777-x) if you prefer paywalled academic articles.

We are going to take the following steps:
 1. open the repository
 2. review the study model
 3. check LCI linking 
 4. compute initial LCIA results
 5. correct model errors
 6. benchmark selected datasets



## Obtain the Study

The study model was submitted to the US Federal LCA Commons. So, we go there to download it.

[![Click "Download"](https://antelopelca.github.io/assets/img/posts/FHWA-1-commons.png)](https://www.lcacommons.gov/lca-collaboration/Federal_Highway_Administration/mtu_pavement/datasets)

We download the full dataset as JSON-LD 2.  Save it on a computer in a location you remember.


In [1]:
PATH_TO_STUDY = '/data/LCI/FedCommons/Federal_Highway_Administration-mtu_pavement.zip'  # edit this to match your system

## Create a catalog
We create a local catalog that is not attached to any local storage.  Be sure to create a `ForegroundCatalog` if you want to do any modeling later.  For the strict benchmarking activities, we could use an `LcCatalog`.

In [2]:
from antelope_foreground import ForegroundCatalog
from antelope import enum

In [3]:
cat = ForegroundCatalog()

Loading JSON data from /data/GitHub/Antelope/core/antelope_core/archives/data/elcd_reference_quantities.json:
local.qdb: /data/GitHub/Antelope/core/antelope_core/archives/data/elcd_reference_quantities.json
local.qdb: Setting NSUUID (False) 77833297-6780-49bf-a61a-0cb707dce700
local.qdb: /data/GitHub/lca-tools/lcatools/qdb/data/elcd_reference_quantities.json
29 total quantity entities added (29 new)
6 total flow entities added (6 new)


## Connect the study
We add a new resource that connects the study and assigns it an origin.

To do this, you need to know what interfaces the dataset supports.  In Antelope, different resources provide different types of information about the LCA data. Here's what each interface provides:
 - `basic`: documentary information (properties)
 - `exchange`: data about process inventories and exchange values
 - `quantity`: data about flow properties and characterization.

Generally OpenLCA datasets include all three.

> Idea! Maybe different data source types should have sensible default interface values.

In [4]:
help(cat.new_resource)

Help on method new_resource in module antelope_foreground.foreground_catalog:

new_resource(reference, source, ds_type, store=True, **kwargs) method of antelope_foreground.foreground_catalog.ForegroundCatalog instance
    Create a new data resource by specifying its properties directly to the constructor
    :param reference:
    :param source:
    :param ds_type:
    :param interfaces: string or tuple of valid interfaces. Defaults to just 'basic' for now.
    :param store: [True] permanently store this resource (disabled for rootless catalogs)
    :param kwargs: priority=0, static=False; **kwargs passed to archive constructor
    :return:



In [5]:
fhwa = cat.new_resource('my.fhwa', PATH_TO_STUDY, ds_type='OpenLcaJsonLdArchive',
                        interfaces=('basic', 'exchange', 'quantity'))

In [6]:
cat.show_interfaces()

local.qdb [basic, index, quantity]
my.fhwa [basic, exchange, quantity]


Then we can interact with the resource by querying it.  


In [7]:
q = cat.query('my.fhwa')

QQQQQQQQQQQQQQQQQQ my.fhwa QQQQQQQQQQQQQQQQQQ
Found Antelope providers:
AntelopeMeta:_dev
antelope_core.providers:IlcdArchive
antelope_core.providers:IlcdLcia
antelope_core.providers:EcospoldV2Archive
antelope_core.providers:EcospoldV1Archive
antelope_core.providers:EcoinventLcia
antelope_core.providers:OpenLcaJsonLdArchive
antelope_core.providers:Traci21Factors
antelope_core.providers:XdbClient
antelope_core.providers:OpenLcaRefData
antelope_background.providers:TarjanBackground
antelope_background.providers:Background
antelope_foreground.providers:AntelopeV1Client
antelope_foreground.providers:LcForeground
antelope_foreground.providers:OryxClient
my.fhwa: /data/LCI/FedCommons/Federal_Highway_Administration-mtu_pavement.zip
my.fhwa: Setting NSUUID (False) None
Found Extension: zip


At this point we can retrieve specific datasets if we know their IDs, but we can't do search or discovery because the data source is not yet indexed.  So, we can search on the Federal LCA commons and identify datasets to retrieve:


In [8]:
p = q.get('72d5a381-8cae-4e1d-b0a3-26cc43b69867')

In [9]:
p.show()

ProcessRef catalog reference (72d5a381-8cae-4e1d-b0a3-26cc43b69867)
origin: my.fhwa
   UUID: 72d5a381-8cae-4e1d-b0a3-26cc43b69867
   Name: Asphalt binder, 8% ground rubber tire (GRT), consumption mix, at terminal, from crude oil, 8% ground rubber tire
Comment: 
Exchange ID 629 (e10e86a1-fd6a-3ac5-a822-55762e8ae99d): Conversion Error from unit kBq to kg
Exchange ID 1053 (bdff254b-ff47-305e-99ee-ff2dddd46876): Conversion Error from unit kBq to kg
==Local Fields==
        Classifications: ['42: Wholesale Trade', '4247: Petroleum and Petroleum Products Merchant Wholesalers']
           SpatialScope: Northern America
          TemporalScope: {'begin': '2015-12-31-05:00', 'end': '2022-12-31-05:00'}
                  @type: Process
            description: This cradle-to-gate dataset covers all relevant process steps and technologies for production of asphalt binder with high overall data quality. The inventory is based on primary data from twelve refineries and eleven terminals in North Amer

Moreover, the data source by itself is not sufficient to perform LCI or LCIA operations because those require a linked technology matrix. In Antelope, this is delivered via the **background** interface, which depends on an **index** of the data.


## Add an index interface 
First, we index the data source- this means loading all its data. An index includes a list of all processes, flows, quantities, and contexts referenced in the data, as well as the reference exchanges for each process.


In [10]:
cat.index_ref(q.origin)

Loading /data/LCI/FedCommons/Federal_Highway_Administration-mtu_pavement.zip
my.fhwa.index.20250807: None
my.fhwa.index.20250807: Setting NSUUID (None) d801437f-b5de-4e88-9f38-494552f874f1
Ignoring ns_uuid specification
my.fhwa: /data/LCI/FedCommons/Federal_Highway_Administration-mtu_pavement.zip
289 total quantity entities added (289 new)
5215 total flow entities added (5215 new)
1298 total process entities added (1298 new)


'my.fhwa.index.20250807'

Now we can see that an index interface was added to the catalog:


In [11]:
cat.show_interfaces()

local.qdb [basic, index, quantity]
my.fhwa [basic, exchange, quantity]
my.fhwa.index.20250807 [basic, index]


Now we can do things like counting and searching. We can also investigate linking the database for LCI computation.


## Investigate Background Linking
The Antelope [Background engine](https://github.com/AntelopeLCA/background) performs Tarjan ordering of Exchange data to detect strongly-connected components (i.e. collections of processes where everything depends on everything). Building this network relies on every *dependent* exchange being linked to a reference exchange of another process. The linking algorithm has a few ways to determine an appropriate provider.

1. In most data sources, an exchange can have a "Preferred provider" (OpenLCA) or "ActivityLink" (ecospold v2) made explicit. These are always followed, as long as they are valid.
2. Some flows may have only one viable *target*, i.e. there is precisely one database process that provides the given flow as a reference. 
3. Some flows may have *no* viable targets, in which case they become cutoffs (like emissions, but into the modeling environment instead of the natural environment)
4. When flows have more than one viable target, the user must specify a preferred provider for each ambiguous flow, or else specify an algorithmic approach to pick one. At present, the only algorithmic choices available are "first" and "last" (alphabetically), "cutoff", or "abort" (the default).

We can learn a lot about a dataset by inspecting its linking characteristics. There's a tool for this called `CheckTerms`. The tool requires a query with the index and exchange interfaces, and inspects each exchange for valid terminations.  For our FHWA database it tells us the following: 


In [12]:
from antelope_core.archives import CheckTerms

In [13]:
check = CheckTerms(q)

1298 processes
1485 reference exchanges
260476 dependent exchanges:
  anchored: 6952 exchanges
  cutoff: 4832 exchanges
  elementary: 248663 exchanges
  self: 16 exchanges

  broken: 11 exchanges
  ambiguous: 2 exchanges


We see that most of the exchanges are proper, but several are "broken" and a few are "ambiguous".  A broken exchange is one in which the specified provider does not provide the linked flow.  Antelope is strict about this relationship- an exchange can only be linked to a process that produces the exact same flow.  

The main reason for this is to prevent errors. However, this is a design decision and could be revisited.  As we will see later in this post, other LCA software does permit squishing providers into exchanges even if the flows don't match.

> Maybe we should consider permitting squishy linking in the background?


### Review broken exchanges

Let's take a look at the broken exchanges:


In [14]:
check.show_broken()

Process: [my.fhwa] Metal composite material (MCM) sheet, at plant [Northern America]
  Disposal, solid waste, unspecified, to inert material landfill <--# ! Petroleum refining, at refinery [Northern America] (0)

Process: [my.fhwa] Natural soda ash (Sodium carbonate), at plant [United States]
  Hazardous waste, DK ==># ! Natural gas, processed, for energy use, at plant [GLO] (0)

Process: [my.fhwa] Calcium carbonate, ground, screened grade, at plant [United States]
  Disposal, mineral waste, underground deposit <--# ! Petroleum refined, for material use, at plant [GLO] (0)

Process: [my.fhwa] Calcium carbonate, ground, fine treated, 3 micron, at plant [United States]
  Lubricant feedstock, at refinery <--# ! Petroleum refined, for material use, at plant [GLO] (1)

Process: [my.fhwa] Calcium carbonate, ground, fine slurry, 3 micron, at plant [United States]
  Transport, combination truck, diesel powered <--# ! Petroleum refined, for material use, at plant [GLO] (1)

Process: [my.fhwa] A

The way to read this output is as follows: The broken exchanges are grouped by process.  Within each process, the broken flow is shown with its direction, with the non-matching target indicated with a !. Then in the parentheses at the end of each line is the number of *valid* targets for the flow in the database.

So the first entry:
```text
Process: [my.fhwa] Metal composite material (MCM) sheet, at plant [Northern America]
  Disposal, solid waste, unspecified, to inert material landfill <--# ! Petroleum refining, at refinery [Northern America] (0)
```

Shows us that the process named 'Metal composite material (MCM) sheet, at plant' has an *inflow* of "Disposal, solid waste, unspecified, to inert material landfill", and it's supposedly being *provided* by "Petroleum refining, at refinery". That is so screwy it must be a mistake.  Let's check the source...

[![Click "Download"](https://antelopelca.github.io/assets/img/posts/FHWA-2-broken.png)](https://www.lcacommons.gov/lca-collaboration/Federal_Highway_Administration/mtu_pavement/dataset/PROCESS/08e766d4-ec5f-3426-a0d4-9533e55f9081)

Indeed, that is a broken link.  Several Federal Commons datasets (including USLCI) had this problem for awhile. Most of them have been fixed, but USLCI was duplicated into the FHWA repository so the problem persists here.


### The art of ignoring errors
Fortunately for us, most of these errors are trivial, and the rest are not too important because they originate in the model *foreground* so we can solve them with modeling later on.

First, the trivial errors: those are the ones with trailing `(0)` or `(1)`.  See, the default behavior of the linker when encountering a broken exchange is to ignore the faulty target and attempt to find a valid target. For the flows with "0" valid targets, those cannot be linked in the current database and will simply become cutoffs.  The ones with "1" valid target will simply be linked to the valid target. So both those sets of errors are trivial.

The only nontrivial errors are the ones with ambiguous matches. These include the broken exchanges with more than one valid target (marked with a `*` above), along with the flows which do not have a target provider specified and have more than one valid target.

In [15]:
amb = enum(check.ambiguous_flows)

 [00] [my.fhwa] Waste, industrial [kg]
 [01] [my.fhwa] Aggregate [kg]
 [02] [my.fhwa] Electricity, AC, 2300-7650 V [MJ]


In [16]:
check.show_ambiguous()

Process: [my.fhwa] Corn steep liquor [Northern America]
* Waste, industrial ==># ! [5622: Waste Treatment and Disposal] (2)

Process: [my.fhwa] Asphalt mix 1 - virgin mix - with EPD Aggregate [GLO]
* Aggregate <--# ! [Asphalt Mixture] (5)


The 'Waste, industrial' flow is also an error, in fact. Look at the targets for that flow:

In [24]:
_=enum(amb[0].targets())

 [00] [my.fhwa] Metal panel, insulated, at plant [Northern America]
 [01] [my.fhwa] Coil, coating, m2, at plant [Northern America]


Neither of those are actual providers for industrial waste management. Those should be designated as "cutoff" in any case. 

The other two flows both belong to the model foreground- we will want to specify those anchors manually, which we will do during modeling.

We do an end-run around the linking problem by telling the linking algorithm to simply set ambiguous link targets to "cutoff".

First, we create a background interface:

In [17]:
cat.background_for_origin('my.fhwa')

QQQQQQQQQQQQQQQQQQ my.fhwa.index.20250807 QQQQQQQQQQQQQQQQQQ
my.fhwa.index.20250807: my.fhwa.index.20250807_background.mat
my.fhwa.index.20250807: Setting NSUUID (False) None


Then, we link the background. If we try using the default settings, the attempt will fail:

In [19]:
cat.query('my.fhwa').check_bg()

Creating flat background
flow: 153cadce-ae16-3fa0-9741-0eb91f1c77eb
Ambiguous termination found for Output: [my.fhwa] Waste, industrial [kg]
Termination Error: process 0879167f-989b-3e3b-b9c5-3c07ea0a3f8e: ref_flow 7f49cac3-84a7-345a-abaa-c3671a78c38f, 


LinkingError: unable to create flat background

So, we specify to simply "cutoff" flows with multiple targets:

In [21]:
cat.query('my.fhwa').check_bg(multi_term='cutoff')

Creating flat background
c245a252-0860-41d2-9789-802bab7984ab: Disposal, wastewater treatment plant residuals, to uns. beneficial use [Input]: Target 4cb0c558-b0ac-3656-bc1f-95b477c14921 MISSING REFERENCE
c245a252-0860-41d2-9789-802bab7984ab: Unspecified polymer [Input]: Target 0aaf1e13-5d80-37f9-b7bb-81a6b8965c71 MISSING REFERENCE
08e766d4-ec5f-3426-a0d4-9533e55f9081: Disposal, solid waste, unspecified, to inert material landfill [Input]: Target 0aaf1e13-5d80-37f9-b7bb-81a6b8965c71 MISSING REFERENCE
0d95cc8b-a9a0-3630-a760-1ab4d88257d8: Hazardous waste, DK [Output]: Target d3db6ab2-33de-453b-ac74-0567ba7fa95b MISSING REFERENCE
247aa74a-8368-37b1-a090-96a1428ef30f: Disposal, mineral waste, underground deposit [Input]: Target 70d2004a-9f80-48ce-a86c-50e85dd6a637 MISSING REFERENCE
34487c89-62dd-3d4e-98dd-06f0c382cc17: Lubricant feedstock, at refinery [Input]: Target 70d2004a-9f80-48ce-a86c-50e85dd6a637 MISSING REFERENCE
48be22f1-e30c-3e4a-8832-a9c5ed32350b: Transport, combination truck, 

True

We see a lot of messages there about missing references-- None of those are unexpected, and they all become cutoffs.

At this point, we have a working background interface, and we can perform LCIA:

In [22]:
cat.show_interfaces()

local.qdb [basic, index, quantity]
my.fhwa [basic, exchange, quantity]
my.fhwa.index.20250807 [background, basic, index]


## Perform LCIA
We will use the free QDB service provided by `vault.lc` to access ReCiPe and do LCIA.

In [25]:
cat.blackbook_guest('https://sc.vault.lc')

GET https://sc.vault.lc/auth/guest.. 

ConnectionError: HTTPSConnectionPool(host='sc.vault.lc', port=443): Max retries exceeded with url: /auth/guest (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7065e02bd0c0>: Failed to establish a new connection: [Errno -3] Temporary failure in name resolution'))

In [29]:
list(cat.blackbook_origins)

GET https://sc.vault.lc/origins.. 200 [1.09 sec]


['lcacommons.fhwa.asphaltframework',
 'lcacommons.useeio.2.0.1',
 'lcacommons.uslci.fy21.q1',
 'lcacommons.uslci.fy22.q3',
 'lcacommons.uslci.fy22.q4.01',
 'lcacommons.uslci.fy23.q4',
 'lcacommons.uslci.fy24.q1.01',
 'lcia.openlca.2.1.4',
 'lcia.traci.2.1']

In [31]:
cat.get_blackbook_resources('lcia.openlca.2.1.4')

GET https://sc.vault.lc/origins/lcia.openlca.2.1.4/resource.. 200 [4.60 sec]


[LcResource(lcia.openlca.2.1.4, dataSource=https://bk.vault.lc/:XdbClient, ['quantity', 'index', 'basic'] [50])]

In [32]:
lcia = cat.query('lcia.openlca.2.1.4')

QQQQQQQQQQQQQQQQQQ lcia.openlca.2.1.4 QQQQQQQQQQQQQQQQQQ
GET https://bk.vault.lc/origins.. 200 [0.19 sec]
lcia.openlca.2.1.4: https://bk.vault.lc/
lcia.openlca.2.1.4: Setting NSUUID (False) None
GET https://bk.vault.lc/lcia.openlca.2.1.4/config.. 200 [0.30 sec]


In [33]:
rs = enum(lcia.lcia(method='recipe'))

GET https://bk.vault.lc/lcia.openlca.2.1.4/lcia.. 200 [0.07 sec]
 [00] [lcia.openlca.2.1.4] Pfister et al 2010 (ReCiPe) [0]
 [01] [lcia.openlca.2.1.4] ReCiPe 2016 Endpoint (E) [0]
 [02] [lcia.openlca.2.1.4] ReCiPe 2016 Endpoint (H) [0]
 [03] [lcia.openlca.2.1.4] ReCiPe 2016 Endpoint (I) [0]
 [04] [lcia.openlca.2.1.4] ReCiPe 2016 Midpoint (E) [0]
 [05] [lcia.openlca.2.1.4] ReCiPe 2016 Midpoint (H) [0]
 [06] [lcia.openlca.2.1.4] ReCiPe 2016 Midpoint (I) [0]


In [34]:
r = rs[4]

In [37]:
r.show()

QuantityRef catalog reference (ReCiPe 2016 Midpoint (E))
origin: lcia.openlca.2.1.4
   UUID: 47461eb7-ac98-3242-97e8-760434380369
   Name: ReCiPe 2016 Midpoint (E)
Comment: 
referenceUnit: 0
==Local Fields==
          Method: ReCiPe 2016 Midpoint (E)
     Description: Method included in openLCA LCIA method package 2.1.3

Compatible with   
- ecoinvent v3.6, v3.7, v3.8  
- Eugeos   
- Agribalyse v3  
- Agrifootprint v5  
- OzLCI    

Method checked against SimaPro 9.1.0.7 (before inclusion of ecoinvent 3.8 flows and characterization factors)
        Category: openLCA LCIA methods 2_1_4
ImpactCategories: [<class 'list'> with 18 entries]
  UnitConversion: {'0': 1.0}
        Synonyms: []
blackbook_origin: lcia.openlca.2.1.4


In [38]:
asph = enum(q.processes(name='asphalt'))

 [00] [fedcommons.fhwa] Asphalt mix 2 - 15% RAP, 3% RAS virgin liquid asphalt binder [GLO]
 [01] [fedcommons.fhwa] Asphalt mix 1 - virgin mix with SBS [GLO]
 [02] [fedcommons.fhwa] Asphalt binder, 0.5% polyphosphoric acid (PPA), consumption mix, at terminal, from crude oil, 0.5% polyphosphoric acid [Northern America]
 [03] [fedcommons.fhwa] Asphalt mix 1 - virgin mix [GLO]
 [04] [fedcommons.fhwa] Asphalt virgin mix, 5% binder, using SBS - variable x% [GLO]
 [05] [fedcommons.fhwa] Asphalt mix 2 - 15% RAP, 3% RAS liquid asphalt binder with SBS [GLO]
 [06] [fedcommons.fhwa] Portable asphalt mix [GLO]
 [07] [fedcommons.fhwa] Asphalt binder, 8% ground rubber tire (GRT), consumption mix, at terminal, from crude oil, 8% ground rubber tire [Northern America]
 [08] [fedcommons.fhwa] Asphalt binder, x% ground rubber tire (GRT), consumption mix, at terminal, from crude oil, x% ground rubber tire [Northern America]
 [09] [fedcommons.fhwa] Asphalt virgin mix, 5% binder, using GRT - variable x%  [GL

In [39]:
asph[11].show()

ProcessRef catalog reference (c86d1b3b-731b-4880-82ad-976b3796c502)
origin: fedcommons.fhwa
   UUID: c86d1b3b-731b-4880-82ad-976b3796c502
   Name: Asphalt mixture - Framework 
Comment: 
reference: [ Asphalt mixture - Framework  [US] ]*==>  907 (kg) Asphalt Mix 
==Local Fields==
        Classifications: ['Asphalt Mixture_Parameterized', 'Fundamental Construct for Asphalt Mixture']
           SpatialScope: US
          TemporalScope: {}
                  @type: Process
            description: This asphalt mixture process is the base case for the asphalt mixtures LCA.

This is a parameterized inventory. 
When using this please note:
- the paramter Total_amount_of_binder indicates the target amount = virgin binder + binder derived from RAP/RAS
- to define a mix, %RAP or %RAS and target asphalt binder %
- fuels for production at plant included in this inventory are: natural gas, diesel, propane and recycled oil, with default values included (Source: NAPA EPD program). Bio-diesel and residu

In [60]:
a['comment'] += '\nHello there'

In [61]:
a.show()

ProcessRef catalog reference (c86d1b3b-731b-4880-82ad-976b3796c502)
origin: fedcommons.fhwa
   UUID: c86d1b3b-731b-4880-82ad-976b3796c502
   Name: Asphalt mixture - Framework 
Comment: 
Hello there
reference: [ Asphalt mixture - Framework  [US] ]*==>  907 (kg) Asphalt Mix 
==Local Fields==
        Classifications: ['Asphalt Mixture_Parameterized', 'Fundamental Construct for Asphalt Mixture']
           SpatialScope: US
          TemporalScope: {}
                  @type: Process
            description: This asphalt mixture process is the base case for the asphalt mixtures LCA.

This is a parameterized inventory. 
When using this please note:
- the paramter Total_amount_of_binder indicates the target amount = virgin binder + binder derived from RAP/RAS
- to define a mix, %RAP or %RAS and target asphalt binder %
- fuels for production at plant included in this inventory are: natural gas, diesel, propane and recycled oil, with default values included (Source: NAPA EPD program). Bio-diese

In [59]:
a['brokenExchanges']

[{'@type': 'Exchange',
  'amount': 0.00016,
  'amountFormula': 'combustion_of_Diesel_Industrial_Equipment ',
  'internalId': 21,
  'defaultProvider': {'@type': 'Process',
   '@id': 'd6ad7035-5498-3237-8abd-50e93b1eef89',
   'name': 'Diesel, combusted in industrial equipment',
   'processType': 'UNIT_PROCESS',
   'location': 'RNA',
   'category': '22: Utilities/2213: Water, Sewage and Other Systems',
   'flowType': 'PRODUCT_FLOW'},
  'flow': {'@type': 'Flow',
   '@id': 'd815ff18-015c-3afb-be18-be03bdf325da',
   'name': 'Diesel, combusted in industrial equipment',
   'flowType': 'PRODUCT_FLOW',
   'location': 'RNA',
   'refUnit': 'm3',
   'category': 'Technosphere Flows/22: Utilities/2213: Water, Sewage and Other Systems'},
  'unit': {'@type': 'Unit',
   '@id': 'ee5f2241-18af-4444-b457-b275660e5a20',
   'name': 'm3*a'},
  'flowProperty': {'@type': 'FlowProperty',
   '@id': '93a60a56-a3c8-22da-a746-0800200c9a66',
   'name': 'Volume',
   'category': 'Technical flow properties'},
  'isAvoid

In [40]:
a = asph[11]

In [42]:
qs = enum(lcia.get(k) for k in r['ImpactCategories'])

GET https://bk.vault.lc/lcia.openlca.2.1.4/05927852-a35a-39d4-a571-7cb54660a561.. 200 [0.05 sec]
 [00] [lcia.openlca.2.1.4] Water consumption [m3] [ReCiPe 2016 Midpoint (E)]
GET https://bk.vault.lc/lcia.openlca.2.1.4/2ff035a8-ca2a-30e9-a0de-79673c497d94.. 200 [0.06 sec]
 [01] [lcia.openlca.2.1.4] Global warming [kg CO2 eq] [ReCiPe 2016 Midpoint (E)]
GET https://bk.vault.lc/lcia.openlca.2.1.4/3b69b396-7853-3de1-9576-87c3abe73c01.. 200 [0.05 sec]
 [02] [lcia.openlca.2.1.4] Marine ecotoxicity [kg 1,4-DCB] [ReCiPe 2016 Midpoint (E)]
GET https://bk.vault.lc/lcia.openlca.2.1.4/43b885c7-b250-3002-ac15-4991d8759476.. 200 [0.05 sec]
 [03] [lcia.openlca.2.1.4] Human non-carcinogenic toxicity [kg 1,4-DCB] [ReCiPe 2016 Midpoint (E)]
GET https://bk.vault.lc/lcia.openlca.2.1.4/55a4d003-bb2a-3326-b6c1-d6a964f1a8d6.. 200 [0.05 sec]
 [04] [lcia.openlca.2.1.4] Freshwater eutrophication [kg P eq] [ReCiPe 2016 Midpoint (E)]
GET https://bk.vault.lc/lcia.openlca.2.1.4/55b7910a-049d-379b-891d-74e9480caaf2.. 

In [58]:
a.bg_lcia(qs[1]).total()/(1-.0107)

0.04814390028302809

In [56]:
qs[1]['quell_biogenic_co2'] = True

In [50]:
res.total()

0.0004266138541360649